In [1]:
import logging

import pandas as pd

import data.helpers as dh
import src.cfr.cfr_helpers as cfrh

import data.cfr_data_19_23 as cfrd
import data.breathe_data as bd
import datetime

Exploring the CF Trust registry data from 2019 to 2023 to evaluate if model output (synthesizing FEV1, FEF25-75 on 2 days) can be used to improve ML achieved from each indidivually 

# Load, process, save yearly data

In [2]:
# Load 2019 data
df19 = cfrd.build_cfr_df(2019)

INFO:root:Loaded {df.shape[0]} entries
INFO:root:2046 entries after removing <18yr


In [4]:
df19.to_excel(
    dh.get_path_to_main() + "ExcelFiles/CFR/CF_Registry_19_processed.xlsx",
    index=False,
)

In [2]:
df23 = cfrd.build_cfr_df(2023)

INFO:root:Loaded 10344 entries
INFO:root:5065 after removing all NaN
INFO:root:2701 entries after removing <18yr


In [7]:
df23.to_excel(
    dh.get_path_to_main() + "ExcelFiles/CFR/CF_Registry_23_processed.xlsx",
    index=False,
)

# Load data

## Link 2019 with 2023 data

In [ ]:
df19 = bd.load_meas_from_excel("CF_Registry_19_processed", study_folder="CFR")

INFO:root:* Checking for same day measurements *


In [9]:
df23

,ID,Age,Height,FEV1,FEF2575,Sex,Date Recorded,ecFEV1,ecFEF2575,ecFEF2575%ecFEV1,Predicted FEV1,ecFEV1 % Predicted,FEV1 % Predicted
0,B155916,36,164.0,1.90,0.75,Female,2023-01-01,1.90,0.75,39.473685,3.166484,60.003459,60.003459
1,B155917,50,174.0,2.76,1.18,Female,2023-01-01,2.76,1.18,42.753621,3.195502,86.371404,86.371404
2,B155918,38,193.0,4.78,3.35,Female,2023-01-01,4.78,3.35,70.083677,4.409788,108.395228,108.395228
3,B155920,42,171.0,4.51,4.08,Female,2023-01-01,4.51,4.08,90.465626,3.304481,136.481353,136.481353
4,B155921,38,150.0,1.37,0.59,Female,2023-01-01,1.37,0.59,43.065691,2.583624,53.026292,53.026292
...,...,...,...,...,...,...,...,...,...,...,...,...,...
10295,C225784,22,185.0,4.90,6.06,Female,2023-01-01,4.90,6.06,123.673466,4.366308,112.222952,112.222952
10300,C225840,25,169.0,3.06,2.74,Female,2023-01-01,3.06,2.74,89.542486,3.563364,85.873907,85.873907
10304,C225865,56,174.0,1.80,0.73,Female,2023-01-01,1.80,0.73,40.555558,3.013003,59.741070,59.741070
10317,C225974,32,180.0,3.24,2.48,Female,2023-01-01,3.24,2.48,76.543210,3.950008,82.025146,82.025146


In [17]:
df23

,ID,Age,Height,FEV1,FEF2575,Sex,Date Recorded,ecFEV1,ecFEF2575,ecFEF2575%ecFEV1,Predicted FEV1,ecFEV1 % Predicted,FEV1 % Predicted
0,B155916,36,164.0,1.90,0.75,Female,2023-01-01,1.90,0.75,39.473685,3.166484,60.003459,60.003459
1,B155917,50,174.0,2.76,1.18,Female,2023-01-01,2.76,1.18,42.753621,3.195502,86.371404,86.371404
2,B155918,38,193.0,4.78,3.35,Female,2023-01-01,4.78,3.35,70.083677,4.409788,108.395228,108.395228
3,B155920,42,171.0,4.51,4.08,Female,2023-01-01,4.51,4.08,90.465626,3.304481,136.481353,136.481353
4,B155921,38,150.0,1.37,0.59,Female,2023-01-01,1.37,0.59,43.065691,2.583624,53.026292,53.026292
...,...,...,...,...,...,...,...,...,...,...,...,...,...
10295,C225784,22,185.0,4.90,6.06,Female,2023-01-01,4.90,6.06,123.673466,4.366308,112.222952,112.222952
10300,C225840,25,169.0,3.06,2.74,Female,2023-01-01,3.06,2.74,89.542486,3.563364,85.873907,85.873907
10304,C225865,56,174.0,1.80,0.73,Female,2023-01-01,1.80,0.73,40.555558,3.013003,59.741070,59.741070
10317,C225974,32,180.0,3.24,2.48,Female,2023-01-01,3.24,2.48,76.543210,3.950008,82.025146,82.025146


In [ ]:
df = pd.concat([df19, df23]).sort_values("ID")

In [ ]:
(df.groupby("ID").apply(lambda df: len(df)) > 1).sum()

/var/folders/zq/v2r6yn111s3gpdf8lzf72xvw0000gn/T/ipykernel_32318/3045146862.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  (df.groupby('ID').apply(lambda df: len(df)) > 1).sum()


1485

In [78]:
df = bd.load_meas_from_excel("CF_Registry_19_23_processed_with_idx", study_folder="CFR")

INFO:root:* Checking for same day measurements *


In [ ]:
ids_with_2_entries = (
    df["ID"].value_counts()[df["ID"].value_counts() == 2].index.tolist()
)
df = df[df.ID.isin(ids_with_2_entries)]

df.to_excel(
    dh.get_path_to_main()
    + "ExcelFiles/CFR/CF_Registry_19_23_stricly_2_entries_processed_with_idx.xlsx",
    index=False,
)

## Use 2019 / 2023 data only

In [ ]:
cols2read = [
    "s01caseid_original",
    "s03clibestfev1",
]
colnames = ["ID", "best FEV1"]

df_bestfev1 = cfrd.load_cfr_data(2023, cols2read, colnames)

In [ ]:
df = bd.load_meas_from_excel("CF_Registry_23_processed", study_folder="CFR")
df = df_bestfev1.merge(df, on=["ID"])

# Is the max FEV1 always >= to FEV1?
(df["best FEV1"] >= df["FEV1"]).count() == df.shape[0]

In [ ]:
# df.to_excel(
#     dh.get_path_to_main() + "ExcelFiles/CFR/CF_Registry_23_processed_with_id.xlsx",
#     index=False,
# )

# Prep running inference

## Add obs indices

In [10]:
df = bd.load_meas_from_excel("CF_Registry_19_processed", study_folder="CFR")

INFO:root:* Checking for same day measurements *


In [14]:
# Add indices for model
# height = df.Height.iloc[0]
# age = df.Age.iloc[0]
# sex = df.Sex.iloc[0]
# ar_prior = "uniform"
# ecfev1_noise_model_cpt_suffix = "_std_add_mult_ecfev1"
# ar_fef2575_cpt_suffix = "_ecfev1_2_days_model_add_mult_noise"
# (
#     HFEV1,
#     uFEV1,
#     ecFEV1,
#     AR,
#     ecFEF2575prctecFEV1,
# ) = var_builders.fev1_fef2575_point_in_time_model_noise_shared_healthy_vars(
#     height,
#     age,
#     sex,
#     ar_prior,
#     ecfev1_noise_model_cpt_suffix,
#     ar_fef2575_cpt_suffix,
# )

import src.models.helpers as mh

ecFEV1 = mh.VariableNode("ecFEV1 (L)", 0, 6, 0.05, prior=None)
ecFEF2575prctecFEV1 = mh.VariableNode("ecFEF25-75 % ecFEV1 (%)", 0, 200, 2, prior=None)

# df[f"idx {ecFEV1.name}"] = df.apply(
#     lambda row: ecFEV1.get_bin_idx_for_value(row["ecFEV1"]), axis=1
# )
# df[f"idx {ecFEF2575prctecFEV1.name}"] = df.apply(
#     lambda row: ecFEF2575prctecFEV1.get_bin_idx_for_value(row["ecFEF2575%ecFEV1"]),
#     axis=1,
# )
df[f"idx FEV1"] = df.apply(
    lambda row: ecFEV1.get_bin_idx_for_value(row["ecFEV1"]), axis=1
)
df[f"idx FEF2575%FEV1"] = df.apply(
    lambda row: ecFEF2575prctecFEV1.get_bin_idx_for_value(row["ecFEF2575%ecFEV1"]),
    axis=1,
)
df[f"idx best FEV1"] = df.apply(
    lambda row: ecFEV1.get_bin_idx_for_value(row["best FEV1"]), axis=1
)

## Custom inference (for 2019 or 2023 only data)

In [15]:
import src.models.helpers as mh

In [16]:
print(f"Initial shape {df.shape}")
df.dropna(subset=["FEV1", "FEF2575", "best FEV1"])
print(f"Final shape {df.shape} - Any rows dropped?")

Initial shape (2046, 17)
Final shape (2046, 17) - Any rows dropped?


In [17]:
df.columns

Index(['ID', 'best FEV1', 'Age', 'Height', 'FEV1', 'FEF2575', 'Sex',
       'Date Recorded', 'ecFEV1', 'ecFEF2575', 'ecFEF2575%ecFEV1',
       'Predicted FEV1', 'ecFEV1 % Predicted', 'FEV1 % Predicted', 'idx FEV1',
       'idx FEF2575%FEV1', 'idx best FEV1'],
      dtype='object')

In [18]:
# Exact inference on 2 days model. Day 1: FEV1, FEF25-75. Day 2: solely FEV1
AR = mh.VariableNode("Airway resistance (%)", 0, 90, 2, prior={"type": "uniform"})
df[AR.name] = df.apply(cfrh.run_ve, axis=1)

In [19]:
df.to_excel(
    dh.get_path_to_main() + "ExcelFiles/CFR/AR_19_data_with_best_FEV1.xlsx",
    index=False,
)

# Load IV variables

In [ ]:
cols2read = [
    "s01caseid_original",
    "s02hospivqty",
    "s02homeivqty",
    "s01coursesoforalantibiotics",
]
colnames = ["ID", "Hosp IVs", "Home IVs", "Oral"]

df = pd.DataFrame(columns=colnames + ["Date Recorded"])
years = [2019, 2020, 2021, 2022, 2023]
for year in years:
    dftmp = cfrd.load_cfr_data(f"{year}", cols2read, colnames)
    dftmp["Date Recorded"] = datetime.date(year, 1, 1)

    df = pd.concat([df, dftmp], ignore_index=True)

/var/folders/zq/v2r6yn111s3gpdf8lzf72xvw0000gn/T/ipykernel_60760/770278454.py:10: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, dftmp], ignore_index=True)


In [9]:
df.head()

,ID,Hosp IVs,Home IVs,Date Recorded,Oral
0,B155916,2.0,1.0,2019-01-01,3.0
1,B155917,0.0,0.0,2019-01-01,2.0
2,B155918,1.0,1.0,2019-01-01,3.0
3,B155920,0.0,0.0,2019-01-01,0.0
4,B155921,1.0,1.0,2019-01-01,3.0


In [10]:
# Ensure uniqueness on ID and Date Recorded
if df.duplicated(subset=["ID", "Date Recorded"]).sum() != 0:
    raise ValueError

print("Number of NaN in 'Hosp IVs':", df["Hosp IVs"].isna().sum())
print("Number of NaN in 'Home IVs':", df["Home IVs"].isna().sum())
print("Number of NaN in 'Oral IVs':", df["Oral"].isna().sum())

df.describe()

Number of NaN in 'Hosp IVs': 33
Number of NaN in 'Home IVs': 35
Number of NaN in 'Oral IVs': 153


,Hosp IVs,Home IVs,Oral
count,50546.000000,50544.000000,50426.000000
mean,0.472995,0.292418,1.710348
std,1.075392,0.864113,2.037367
min,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000
50%,0.000000,0.000000,1.000000
75%,1.000000,0.000000,3.000000
max,20.000000,15.000000,71.000000


In [ ]:
df["Any antibiotics"] = df["Hosp IVs"] + df["Home IVs"] + df["Oral"]

In [15]:
df.head()

,ID,Hosp IVs,Home IVs,Date Recorded,Oral,Any antibiotics
0,B155916,2.0,1.0,2019-01-01,3.0,6.0
1,B155917,0.0,0.0,2019-01-01,2.0,2.0
2,B155918,1.0,1.0,2019-01-01,3.0,5.0
3,B155920,0.0,0.0,2019-01-01,0.0,0.0
4,B155921,1.0,1.0,2019-01-01,3.0,5.0


In [ ]:
# Check number of IVs, now showing normalized percentage per bar and showing the percent label per bar
import plotly.express as px


# for col in ["Hosp IVs", "Home IVs"]:
for col in ["Any antibiotics", "Hosp IVs", "Home IVs", "Oral"]:
    fig = px.histogram(df, x=col, histnorm="percent", text_auto=".2f")
    fig.update_traces(textangle=270)
    ids = df[~df[col].isna()].ID.nunique()
    notnan = (~df[col].isna()).sum()
    title = f"{col} from 2019-23 registry, {notnan} entries, {ids} IDs"
    fig.update_layout(
        height=300,
        width=800,
        title=dict(text=title, font=dict(size=14)),
        margin=dict(l=40, r=20, t=40, b=40),
        xaxis_title=col,
        yaxis_title="Percent",
    )
    fig.show()
    # fig.write_image(f"{dh.get_path_to_main()}PlotsCFR/{title}.pdf")

# CCL: 75% of individuals had no hosp IVs between 2019 and 2023. 85% no home IVs. Trikata arrived in nov 2020

In [ ]:
# Compute avg IVs per year
# skipna is True by default meaning that nans are excluded
df_avg_ivs_per_year = (
    df.groupby("ID")
    .agg(
        {
            "Home IVs": "mean",
            "Hosp IVs": "mean",
            "Oral": "mean",
            "Any antibiotics": "mean",
        }
    )
    .rename(columns={"Home IVs": "Avg Home IVs", "Hosp IVs": "Avg Hosp IVs"})
)

df_avg_ivs_per_year.describe(percentiles=[])
# CCL: 2x more hosp IVs than home IVs
# Mean hosp IVs: 0.5 per year

,Avg Home IVs,Avg Hosp IVs,Oral,Any antibiotics
count,11663.000000,11663.000000,11663.000000,11663.000000
mean,0.284902,0.500503,1.750503,2.535817
std,0.688697,0.922580,1.576887,2.244520
min,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.200000,1.400000,2.000000
max,11.000000,20.000000,14.200000,25.000000


In [117]:
df_avg_ivs_per_year

,ID,Avg Home IVs,Avg Hosp IVs,Oral,Any antibiotics
ID,,,,,
0,B155916,0.2,0.4,0.8,1.4
1,B155917,0.0,0.0,1.4,1.4
2,B155918,0.6,0.2,1.4,2.2
3,B155920,0.0,0.2,0.6,0.8
4,B155921,0.2,0.2,1.8,2.2
...,...,...,...,...,...
11658,C226145,0.0,0.0,0.0,0.0
11659,C226146,0.0,0.0,1.0,1.0
11660,C226147,0.0,2.0,7.0,9.0


In [ ]:
# Melt the dataframe to long format with columns "ID", "Avg", "Type"
df_avg_ivs_long = df_avg_ivs_per_year.melt(
    id_vars="ID",
    value_vars=["Avg Home IVs", "Avg Hosp IVs", "Oral"],
    var_name="Type",
    value_name="Avg",
)
df_avg_ivs_long

,ID,Type,Avg
0,B155916,Avg Home IVs,0.2
1,B155917,Avg Home IVs,0.0
2,B155918,Avg Home IVs,0.6
3,B155920,Avg Home IVs,0.0
4,B155921,Avg Home IVs,0.2
...,...,...,...
34984,C226145,Oral,0.0
34985,C226146,Oral,1.0
34986,C226147,Oral,7.0
34987,C226148,Oral,2.0


In [ ]:
import numpy as np

# Determine the max value for binning
max_val = df_avg_ivs_long["Avg"].max()
# Create bin edges: first bin for 0, then (0,1], (1,2], ..., (N-1,N]
# Ensure max_val+1 to include the rightmost data point
bin_edges = np.concatenate([np.arange(-1, np.ceil(max_val) + 1)])
bin_labels = ["0"] + [f"({i};{i+1}]" for i in range(0, int(np.ceil(max_val)))]

antibio_binned = pd.cut(
    df_avg_ivs_long["Avg"],
    bins=bin_edges,
    labels=bin_labels,
)

fig = px.histogram(
    df_avg_ivs_long.assign(antibio_binned=antibio_binned),
    x="antibio_binned",
    color="Type",
    histnorm="percent",
    text_auto=".1f",
    category_orders={"antibio_binned": bin_labels},
)

# Make the first bar (bin for '0') white
bar_colors = ["grey"] + ["#0072b2"] * (len(bin_labels) - 1)
fig.update_traces(
    textangle=90,
    # marker_color=bar_colors,
    textfont=dict(size=9),
    textposition="outside",
)
fig.update_yaxes(range=[0, 150])
fig.update_xaxes(tickangle=45)

title = f"Demography per antibiotic in 2019-23 registry, {ids} IDs"

fig.update_layout(
    height=300,
    width=800,
    title=dict(text=title, font=dict(size=12)),
    margin=dict(l=40, r=20, t=40, b=40),
    xaxis_title="Average number of antibiotics per year (hosp/home IV or oral)",
    yaxis_title="Proportion",
)
fig.show()
fig.write_image(f"{dh.get_path_to_main()}PlotsCFR/IV associations/{title}.pdf")

In [ ]:
import numpy as np

# Determine the max value for binning
max_val = df_avg_ivs_per_year["Any antibiotics"].max()
# Create bin edges: first bin for 0, then (0,1], (1,2], ..., (N-1,N]
# Ensure max_val+1 to include the rightmost data point
bin_edges = np.concatenate([np.arange(-1, np.ceil(max_val) + 1)])
bin_labels = ["0"] + [f"({i};{i+1}]" for i in range(0, int(np.ceil(max_val)))]

antibio_binned = pd.cut(
    df_avg_ivs_per_year["Any antibiotics"],
    bins=bin_edges,
    labels=bin_labels,
)

fig = px.histogram(
    df_avg_ivs_per_year.assign(antibio_binned=antibio_binned),
    x="antibio_binned",
    histnorm="percent",
    text_auto=".1f",
    category_orders={"antibio_binned": bin_labels},
)

# Make the first bar (bin for '0') white
bar_colors = ["grey"] + ["#0072b2"] * (len(bin_labels) - 1)
fig.update_traces(
    textangle=90,
    marker_color=bar_colors,
    textfont=dict(size=9),
    textposition="outside",
)
fig.update_yaxes(range=[0, 29])
fig.update_xaxes(tickangle=45)

title = f"Demography of all antibiotics in 2019-23 registry, {ids} IDs"

fig.update_layout(
    height=300,
    width=800,
    title=dict(text=title, font=dict(size=12)),
    margin=dict(l=40, r=20, t=40, b=40),
    xaxis_title="Average number of antibiotics per year (hosp/home IV or oral)",
    yaxis_title="Proportion",
)
# fig.show()
fig.write_image(f"{dh.get_path_to_main()}PlotsCFR/IV associations/{title}.pdf")

In [114]:
df.to_excel(
    dh.get_path_to_main() + "ExcelFiles/CFR/anbitiotics_data_19-23.xlsx",
    index=False,
)

In [ ]:
year = 2023
df["Date Recorded"] = datetime.date(year, 1, 1)

In [20]:
rename_dict = dict(zip(cols2read, colnames))
df = df.rename(columns=rename_dict)